In [76]:
import pandas as pd
from graph import Graph
from flight import Flight
from airport import Airport
from pnr import PNR
from datetime import datetime
import numpy as np
from dimod import BinaryQuadraticModel
import dimod
import itertools

#import singularity.optimization as sop

In [77]:
import time 
t1 = time.time()

In [78]:
SINGULARITY_TOKEN = "c8b07936-99b0-416f-a749-150da42f6aa1"
#DWAVE_API_TOKEN = "DEV-7b444792d7a75ce33cc91f50fac2e0e021db9259"

#=======================================================================================================
# Create the graph
flight_graph = Graph()

# Path to the CSV file
available_flights_df = 'data_files/PRMI-DM-AVAILABLE_FLIGHTS.csv'  
cancelled_flights_df = 'data_files/PRMI-DM_TARGET_FLIGHTS_test.csv' 

# Build the graph using the CSV file
flight_graph.add_flights_from_csv(available_flights_df)
flight_graph.add_cancelled_flights_from_csv(cancelled_flights_df)

# # Draw the graph
# flight_graph.draw_graph()

In [79]:
# Step 1: Load both CSV files
target_flights_df = pd.read_csv("data_files/PRMI-DM_TARGET_FLIGHTS.csv")
pnr_df = pd.read_csv("data_files/PRMI_DM_ALL_PNRs.csv")
available_flights_df = pd.read_csv("data_files/PRMI-DM-AVAILABLE_FLIGHTS.csv")

In [80]:
print("Reduced dataset!")

#available_flights_df = available_flights_df[ 0 : len(available_flights_df) // 2 ]
#cancelled_flights = cancelled_flights[0:2]
#pnr_df = pnr_df[0:10]
print("Passenger rows: ", len(pnr_df))
print("Flights rows: ", len(available_flights_df))

Reduced dataset!
Passenger rows:  43832
Flights rows:  3221


In [81]:
pnr_df[pnr_df["RECLOC"] == 1]

,RECLOC,CREATION_DTZ,CABIN_CD,COS_CD,OPER_OD_ORIG_CD,OPER_OD_DEST_CD,DEP_KEY,DEP_DT,ORIG_CD,DEST_CD,FLT_NUM,DEP_DTML,ARR_DTML,DEP_DTMZ,ARR_DTMZ,OD_BROKEN_IND,PAX_CNT,CVM,CONN_TIME_MINS
1,1,2027-09-03,Y,21,KHL,RHP,AZ20271021KHLTPH9400,2027-10-21,KHL,TPH,9400,2027-10-21 11:38,2027-10-21 16:29,2027-10-21 14:38,2027-10-21 21:29,0,3,5.68905,NaN
2,1,2027-09-03,Y,20,RHP,KHL,AZ20271104RHPTPH3571,2027-11-04,RHP,TPH,3571,2027-11-04 19:57,2027-11-04 23:25,2027-11-05 0:57,2027-11-05 4:25,0,3,5.68905,NaN
3,1,2027-09-03,Y,20,RHP,KHL,AZ20271104TPHKHL548,2027-11-04,TPH,KHL,548,2027-11-05 0:51,2027-11-05 9:53,2027-11-05 5:51,2027-11-05 12:53,0,3,5.68905,86.0
4,1,2027-09-03,Y,21,KHL,RHP,AZ20271021TPHRHP12465,2027-10-21,TPH,RHP,12465,2027-10-21 18:25,2027-10-21 21:55,2027-10-21 23:25,2027-10-22 2:55,0,3,5.68905,116.0


In [82]:
# Step 2: Differentiate between trips as mentioned in the chat

pnr_df = pnr_df.sort_values(by=["DEP_DTML"], ascending = [True])
grouped = pnr_df.groupby(["RECLOC"])
for rec, group1 in grouped:
    trip_number = 1
    
    grouped2 = group1.groupby(["OPER_OD_ORIG_CD", "OPER_OD_DEST_CD", "DEP_DT"])

    for (orig, dest, date), group2 in grouped2:
        pnr_df.loc[group2.index, "TRIP_NUMBER"] = int(trip_number)
        trip_number = trip_number + 1

In [83]:
# Step 3: Filter PNRs with matching DEP_KEY in both datasets
# Extract the DEP_KEYs from the target flights
target_dep_keys = target_flights_df['DEP_KEY'].unique()

# Filter PNRs where DEP_KEY matches
matching_pnr_df = pnr_df[pnr_df['DEP_KEY'].isin(target_dep_keys)]

In [84]:
# Step 4: Create a list of PNR objects from the filtered PNR DataFrame

# Well what we can do is instead make the list of PASSENGERS (pnr_list) LONGER
# Such that each entrance is different according to the TRIP 
pnr_list_1leg = []
pnr_list_2leg = []
# Matrix saving all of the passengers independently of whether the trip is 1-leg or 2-legged
pnr_list = []

group = matching_pnr_df.groupby(["RECLOC", "TRIP_NUMBER"])

for rec_and_trpnr, group in group:
    recloc = rec_and_trpnr[0]
    trip_number = int( rec_and_trpnr[1] )
    
    # We select the selected pnr
    ## We select from the original PNR in order to differentiate between originally booked as direct or multi-legged
    ## And then re-scheduled as direct (forced or simple) direct
    selected_pnr = pnr_df[(pnr_df["RECLOC"] == recloc) & (pnr_df["TRIP_NUMBER"] == trip_number)] 

    # Now we iterate over all of the rows for this given configuration
    for ktrip, row in group.iterrows():
       #trip_id = f"{'1'}_{'2'}_{legs}legs_{'3'}"
       # We include the trip number and the number of legs
        trip_id = f"{row['RECLOC']}_{int(row['TRIP_NUMBER'])}_{row['DEP_KEY']}"
        pnr = PNR(
            recloc=row['RECLOC'],
            creation_dtz=row['CREATION_DTZ'],
            cabin_cd=row['CABIN_CD'],
            cos_cd=row['COS_CD'],
            oper_od_orig_cd=row['OPER_OD_ORIG_CD'],
            oper_od_dest_cd=row['OPER_OD_DEST_CD'],
            dep_key=row['DEP_KEY'],
            dep_dt=row['DEP_DT'],
            orig_cd=row['ORIG_CD'],
            dest_cd=row['DEST_CD'],
            flt_num=row['FLT_NUM'],
            dep_dtml=row['DEP_DTML'],
            arr_dtml=row['ARR_DTML'],
            dep_dtmz=row['DEP_DTMZ'],
            arr_dtmz=row['ARR_DTMZ'],
            od_broken_ind=row['OD_BROKEN_IND'],
            pax_cnt=row['PAX_CNT'],
            cvm=row['CVM'],
            conn_time_mins=row['CONN_TIME_MINS']
        )
        pnr.trip_id = trip_id
        pnr.trip_number = int(trip_number),
        pnr.trip_legs = len(group),

        ## Now we check if the trip was originally booked as multi-legged or not
        ## The trip only consists of one flight
        if ((pnr.oper_od_orig_cd == pnr.orig_cd) & (pnr.oper_od_dest_cd == pnr.dest_cd)):
            pnr.booked_multi_leg = False
            
            # The flight ORIGINALLY consisted of one leg!
            ## Therefore the "ideal" arrival and departure times are the same 
            ## As the leg that is missing
            pnr.ideal_dep_dtmz = pnr.dep_dtmz
            pnr.ideal_dep_dtml = pnr.dep_dtml

            pnr.ideal_arr_dtmz = pnr.arr_dtmz
            pnr.ideal_arr_dtml = pnr.arr_dtml
        else:
            pnr.booked_multi_leg = True
            
            # The flight ORIGINALLY did NOT consist of one leg! 
            ## Therefore the "ideal" arrival and departure times are the ones
            ## corresponding to the first taken leg
            pnr.ideal_dep_dtmz = min(selected_pnr["DEP_DTMZ"])
            pnr.ideal_dep_dtml = min(selected_pnr["DEP_DTML"])

            
            pnr.ideal_arr_dtmz = max(selected_pnr["ARR_DTMZ"])
            pnr.ideal_arr_dtml = max(selected_pnr["ARR_DTML"])            


        if len(group) == 1:
            pnr_list_1leg.append(pnr)
            
            # We add an additional attribute that tell us whether or not
            # there are more than 1 cancelled flights in the trip
            pnr.trip_multi_leg = False
        elif len(group) == 2:
            pnr_list_2leg.append(pnr)

            # We add an additional attribute that tell us whether or not
            # there are more than 1 cancelled flights in the trip
            pnr.trip_multi_leg = True
        
        # We save it to the general pnr_list either way
        pnr_list.append(pnr)


print("Number of passengers whose booked trip has 1 leg: ", len(pnr_list_1leg))
print("Number of passengers whose booked trip has 2 legs: ", len(pnr_list_2leg))

#print("Total number of flights: ", len(pnr_list_1leg) + len(pnr_list_2leg))
print("Total number of flights: ", len(pnr_list))


Number of passengers whose booked trip has 1 leg:  13346
Number of passengers whose booked trip has 2 legs:  1750
Total number of flights:  15096


In [85]:
available_flights_df

,DEP_KEY,DEP_DT,ORIG_CD,DEST_CD,FLT_NUM,DEP_DTML,ARR_DTML,DEP_DTMZ,ARR_DTMZ,C_CAP_CNT,C_AUL_CNT,C_PAX_CNT,C_AVAIL_CNT,Y_CAP_CNT,Y_AUL_CNT,Y_PAX_CNT,Y_AVAIL_CNT
0,AZ20271023RHPTPH17919,2027-10-23,RHP,TPH,17919,2027-10-23 22:09:00,2027-10-24 01:37:00,2027-10-24 03:09:00,2027-10-24 06:37:00,16,16,11,5,138,138,117,21
1,AZ20271021RHPTPH17919,2027-10-21,RHP,TPH,17919,2027-10-21 22:09:00,2027-10-22 01:37:00,2027-10-22 03:09:00,2027-10-22 06:37:00,16,16,14,2,144,144,144,0
2,AZ20271024RHPTPH17919,2027-10-24,RHP,TPH,17919,2027-10-24 22:09:00,2027-10-25 01:37:00,2027-10-25 03:09:00,2027-10-25 06:37:00,16,16,13,3,144,144,122,22
3,AZ20271025RHPTPH17919,2027-10-25,RHP,TPH,17919,2027-10-25 22:09:00,2027-10-26 01:37:00,2027-10-26 03:09:00,2027-10-26 06:37:00,16,16,14,2,144,144,111,33
4,AZ20271027RHPTPH17919,2027-10-27,RHP,TPH,17919,2027-10-27 22:09:00,2027-10-28 01:37:00,2027-10-28 03:09:00,2027-10-28 06:37:00,12,12,11,1,114,114,95,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,AZ20271019TPHALN6524,2027-10-19,TPH,ALN,6524,2027-10-20 03:40:00,2027-10-20 06:37:00,2027-10-20 08:40:00,2027-10-20 12:37:00,16,16,16,0,144,144,144,0
3217,AZ20271020TPHALN6524,2027-10-20,TPH,ALN,6524,2027-10-21 03:40:00,2027-10-21 06:37:00,2027-10-21 08:40:00,2027-10-21 12:37:00,16,16,16,0,144,144,145,-1
3218,AZ20271021TPHALN6524,2027-10-21,TPH,ALN,6524,2027-10-22 03:40:00,2027-10-22 06:37:00,2027-10-22 08:40:00,2027-10-22 12:37:00,16,16,16,0,144,144,144,0
3219,AZ20271028TPHALN6524,2027-10-28,TPH,ALN,6524,2027-10-29 03:40:00,2027-10-29 06:37:00,2027-10-29 08:40:00,2027-10-29 12:37:00,16,16,7,9,144,144,87,57


In [86]:
# We fix the columns to only include minutes
date_columns = ["DEP_DTML", "ARR_DTML", "DEP_DTMZ", "ARR_DTMZ"]

# Convert datetime columns to the correct format
for col in date_columns:
    available_flights_df[col] = pd.to_datetime(available_flights_df[col], errors="coerce", infer_datetime_format=True)
    available_flights_df[col] = available_flights_df[col].dt.strftime("%Y-%m-%d %H:%M")

# Print to check if format is correct
print(available_flights_df[date_columns].head())

           DEP_DTML          ARR_DTML          DEP_DTMZ          ARR_DTMZ
0  2027-10-23 22:09  2027-10-24 01:37  2027-10-24 03:09  2027-10-24 06:37
1  2027-10-21 22:09  2027-10-22 01:37  2027-10-22 03:09  2027-10-22 06:37
2  2027-10-24 22:09  2027-10-25 01:37  2027-10-25 03:09  2027-10-25 06:37
3  2027-10-25 22:09  2027-10-26 01:37  2027-10-26 03:09  2027-10-26 06:37
4  2027-10-27 22:09  2027-10-28 01:37  2027-10-28 03:09  2027-10-28 06:37


In [87]:
available_flights_df.head()

,DEP_KEY,DEP_DT,ORIG_CD,DEST_CD,FLT_NUM,DEP_DTML,ARR_DTML,DEP_DTMZ,ARR_DTMZ,C_CAP_CNT,C_AUL_CNT,C_PAX_CNT,C_AVAIL_CNT,Y_CAP_CNT,Y_AUL_CNT,Y_PAX_CNT,Y_AVAIL_CNT
0,AZ20271023RHPTPH17919,2027-10-23,RHP,TPH,17919,2027-10-23 22:09,2027-10-24 01:37,2027-10-24 03:09,2027-10-24 06:37,16,16,11,5,138,138,117,21
1,AZ20271021RHPTPH17919,2027-10-21,RHP,TPH,17919,2027-10-21 22:09,2027-10-22 01:37,2027-10-22 03:09,2027-10-22 06:37,16,16,14,2,144,144,144,0
2,AZ20271024RHPTPH17919,2027-10-24,RHP,TPH,17919,2027-10-24 22:09,2027-10-25 01:37,2027-10-25 03:09,2027-10-25 06:37,16,16,13,3,144,144,122,22
3,AZ20271025RHPTPH17919,2027-10-25,RHP,TPH,17919,2027-10-25 22:09,2027-10-26 01:37,2027-10-26 03:09,2027-10-26 06:37,16,16,14,2,144,144,111,33
4,AZ20271027RHPTPH17919,2027-10-27,RHP,TPH,17919,2027-10-27 22:09,2027-10-28 01:37,2027-10-28 03:09,2027-10-28 06:37,12,12,11,1,114,114,95,19


In [88]:
#Available_flights_df = 'data_files/PRMI-DM-AVAILABLE_FLIGHTS.csv'  
#Available_flights_df = pd.read_csv(Available_flights_df)

# Now we create the available_flights list
available_flights = []
for _, row in available_flights_df.iterrows():
    flight = Flight(
        dep_key = row["DEP_KEY"],
        dep_dt = row["DEP_DT"],
        orig_cd = row["ORIG_CD"],
        dest_cd = row["DEST_CD"], 
        flt_num = row["FLT_NUM"], 
        dep_dtml = row["DEP_DTML"], 
        arr_dtml = row["ARR_DTML"], 
        dep_dtmz = row["DEP_DTMZ"], 
        arr_dtmz = row["DEP_DTMZ"],
        c_cap_cnt = row["C_CAP_CNT"],
        c_aul_cnt = row["C_AUL_CNT"],
        c_pax_cnt = row["C_PAX_CNT"],
        c_avail_cnt = row["C_AVAIL_CNT"],
        y_cap_cnt = row["Y_CAP_CNT"],
        y_aul_cnt = row["Y_AUL_CNT"],
        y_pax_cnt = row["Y_PAX_CNT"],
        y_avail_cnt = row["Y_AVAIL_CNT"],
        status = True # Put it as to indicate that the flight is available
        # I am not following too much of the code that was previously shared and kind of doing my own
        # Either wa the status wont be used...
    )
    available_flights.append(flight)

In [89]:
print("Total dataset:")
print("Number of passengers: ", len(pnr_list) )
print("Number of available flights: ", len(available_flights) )

Total dataset:
Number of passengers:  15096
Number of available flights:  3221


## Now we compute the flight search space

In [90]:
## We now compute the possiblities for each origin and destination

flights_out_dic = {}
flights_in_dic = {}
flights_out_in_dic = {}
# This last dictionary contains all the possible two-legged combinations for a given origin and destination
flights_2legs_out_in_dic = {}


# We initialize the dictionaries
orig_keys = available_flights_df["ORIG_CD"].unique()
dest_keys = available_flights_df["DEST_CD"].unique()

for orig_key in orig_keys:
    flights_out_dic[str(orig_key)] = []

for dest_key in dest_keys:
    flights_in_dic[str(dest_key)] = []

# Now we initialize for the one-legged and two-legged "Master" dictionaries
#combinations = list( Available_flights_df.groupby(["ORIG_CD", "DEST_CD"]).groups.keys() ) #WRONG!!!
combinations = list(itertools.product(orig_keys, dest_keys))
for combination in combinations:
    flights_out_in_dic[(str(combination[0]), str(combination[1]))] = []
    flights_2legs_out_in_dic[(str(combination[0]), str(combination[1]))] = []

# Now we fill in the DIRECT dictionaries
for flight in available_flights:
    flights_out_dic[str(flight.orig_cd)].append(flight)
    flights_in_dic[str(flight.dest_cd)].append(flight)
    flights_out_in_dic[( str(flight.orig_cd), str(flight.dest_cd) )].append(flight)

In [91]:
available_flights[0]

Flight 17919 (RHP -> TPH):
  Departure Date: 2027-10-23
  Departure Time (Local): 2027-10-23 22:09, (Zulu): 2027-10-24 03:09
  Arrival Time (Local): 2027-10-24 01:37, (Zulu): 2027-10-24 03:09
  Cabin C: Capacity: 16, Occupied: 16, Passengers: 11, Available: 5
  Cabin Y: Capacity: 138, Occupied: 138, Passengers: 117, Available: 21
  Departure Key: AZ20271023RHPTPH17919

In [92]:
# Now we fill the dictionaries for two-legged flights
for combination in combinations:
    print("COMBINATIONS!!! ***************************")
    print("First flight: {}".format(combination[0]))
    print("Second flight: {}".format(combination[1]))
    first_legs = flights_out_dic[ str(combination[0]) ]
    second_legs = flights_in_dic[ str(combination[1]) ]

    # Now we check each combination
    for first_leg in first_legs:
        for second_leg in second_legs:
            
            # if the first and second leg do not satisfy spatial conditions,
            if (first_leg.dest_cd != second_leg.orig_cd):
                continue
            
            # Now we check time conditions
            first_leg_arr = datetime.strptime(first_leg.arr_dtmz, "%Y-%m-%d %H:%M")
            second_leg_dep = datetime.strptime(second_leg.dep_dtmz, "%Y-%m-%d %H:%M")
            conn_time = (second_leg_dep - first_leg_arr).total_seconds() / 60.0    

            ## if the connection time is too low or the connection time is too high
            ### Minimum 60 minutes of time and maximum 12 hours of connection time
            if ((conn_time <= 60) or (720 <= conn_time)):
                continue
            else:
                flights_2legs_out_in_dic[(str(combination[0]), str(combination[1]))].append( (first_leg, second_leg) )
                #print(r"Two-legged valid!! from {} to {}".format(combination[0], combination[1]) )
                #print("Combination: ", (first_leg.dep_key, second_leg.dep_key))
                


print("Flight inventories finished!!! ")

COMBINATIONS!!! ***************************
First flight: RHP
Second flight: TPH
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: NWY
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: JDP
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: QXG
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: TZJ
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: OSW
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: OVS
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: NAD
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: VUY
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: RBR
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: IPF
COMBINATIONS!!! ***************************
First flight: RHP
Second flight: ERW
COMBINATIONS!!! ************

In [93]:
total_single_flights = 0
for key in flights_out_in_dic:
    total_single_flights = total_single_flights + len( flights_out_in_dic[key] )

print("Total flights for one leg: ", total_single_flights)

Total flights for one leg:  3221


In [94]:
total_double_flights = 0
for key in flights_2legs_out_in_dic:
    total_double_flights = total_double_flights + len( flights_2legs_out_in_dic[key] )

print("Total flights for two leg: ", total_double_flights)

Total flights for two leg:  129649


In [95]:
print("Total number of flight variables: ", total_single_flights + total_double_flights)

Total number of flight variables:  132870


In [96]:
len(pnr_list)

15096

In [97]:
print("Total number of possible variables: ", len(pnr_list) * (total_single_flights + total_double_flights) )

Total number of possible variables:  2005805520


raise Exception("Stopping execution here")

In [98]:
## We now define the time penalty computation, 
#### The leg_penalty variable is for us to add to multi-legt flights
#### Those will have a high leg_penalty, as we prefer direct flights
def time_penalty(time_difference: float) -> float:
        if time_difference < 0:
            return -10000
        elif time_difference <= 360:
            #return (70 - time_difference)
            return 70
        elif time_difference <= 720:
            #return (50 - time_difference)
            return 50
        elif time_difference <= 1440:
            #return (40 - time_difference)
            return 40
        elif time_difference <= 2880:
            #return (30 - time_difference)
            return 30
        else:
            return -10000

In [99]:
def negative_time_penalty(time_difference: float) -> float:
        if time_difference < 0:
            return 10000
        elif time_difference <= 360:
            #return (70 - time_difference)
            return -70
        elif time_difference <= 720:
            #return (50 - time_difference)
            return -50
        elif time_difference <= 1440:
            #return (40 - time_difference)
            return -40
        elif time_difference <= 2880:
            #return (30 - time_difference)
            return -30
        else:
            return 10000

In [100]:
# We add costs for the CVM variables

one_one_cost = 25
one_multi_cost = int(15)
multi_one_cost = int(20)
multi_multi_cost = int(10)

# Factor to include in the computation of the cvm!
cvm_factor = {}
cvm_factor["one_one"] = int(25)
cvm_factor["one_multi"] = int(15)
cvm_factor["multi_one"] = int(20)
cvm_factor["multi_multi"] = int(10)
cvm_factor["forced_one"] = int(30)

# To change this cost, depending on the feedback that we obtain
forced_one_cost = 30
## Different penalties for each proposed solution

In [101]:
n_variables = 0
n_one_one = 0
n_one_multi = 0
n_multi_multi = 0
n_multi_one = 0
n_forced_one = 0

In [102]:
print(" ¡Setting seach space! ")

#pnr_list = pnr_list[0:2]
#pnr_list = pnr_list[0:len(pnr_list)//135]

#available_flights = available_flights[0:10]

print("Reduced dataset: ")
print("Number of passengers: ", len(pnr_list) )
print("Number of available flights: ", len(available_flights) )

 ¡Setting seach space! 
Reduced dataset: 
Number of passengers:  15096
Number of available flights:  3221


In [103]:
## Now we create the dictionaries! 
# For local constraints
flights_per_passenger = {}
# For seat constraints!
passengers_per_flight = {}

# We initialize / initialise the variables_per_flight dictionary
for flight in available_flights:
    passengers_per_flight[str(flight.dep_key)] = {}
    passengers_per_flight[str(flight.dep_key)]["one_one"] = []
    passengers_per_flight[str(flight.dep_key)]["forced_one"] = []
    passengers_per_flight[str(flight.dep_key)]["one_multi"] = []
    passengers_per_flight[str(flight.dep_key)]["multi_one"] = []
    passengers_per_flight[str(flight.dep_key)]["multi_multi"] = []


In [104]:
general_dic = {}
general_dic["variables"] = []
general_dic["values"] = []
general_dic["pax_cnt"] = []
general_dic["avail_cnt"] = []

## Now we do different fors for the cases

In [105]:
import copy 

print("Setting up the dictionary!")
for passenger in pnr_list:
    #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))] = general_dic
    #variables_per_passenger[str(passenger.trip_id)] = copy.deepcopy(general_dic)
    flights_per_passenger[str(passenger.trip_id)] = {}
    
    flights_per_passenger[str(passenger.trip_id)]["one_one"] = copy.deepcopy(general_dic)
    flights_per_passenger[str(passenger.trip_id)]["forced_one"] = copy.deepcopy(general_dic)
    flights_per_passenger[str(passenger.trip_id)]["multi_one"] = copy.deepcopy(general_dic)
    flights_per_passenger[str(passenger.trip_id)]["one_multi"] = copy.deepcopy(general_dic)
    flights_per_passenger[str(passenger.trip_id)]["multi_multi"] = copy.deepcopy(general_dic)

Setting up the dictionary!


## One-one case

In [106]:
for passenger in pnr_list:
    #print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))
    #print("Passenger trip id: ", passenger.trip_id)
    
    if (not passenger.booked_multi_leg):
        available_flights_case = flights_out_in_dic[ (str(passenger.orig_cd), str(passenger.dest_cd)) ]
        
        # Now we iterate over all possible case
        for flight in available_flights_case:

            # Retrieve the new time departures
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 
            
            # Retrieve the original time departures
            orig_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
            orig_time_arr = datetime.strptime(passenger.arr_dtmz, "%Y-%m-%d %H:%M")
                
            time_diff_dep = ((new_time_dep - orig_time_dep).total_seconds() / 60.0) # Minutes
            time_diff_arr = ((new_time_arr - orig_time_arr).total_seconds() / 60.0) 

            cost_dep = time_penalty(time_diff_dep)
            cost_arr = time_penalty(time_diff_arr)

            # Is this a good pairing?
            if ((cost_dep <= 0) or (cost_arr <= 0)): # Check the penalty
                #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:
                var_cost = cost_dep + cost_arr + cvm_factor["one_one"]*passenger.cvm
                
                # We save the variable

                # We save the assigned DIRECT flight for this passenger
                #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )
                flights_per_passenger[str(passenger.trip_id)]["one_one"]["variables"].append((passenger.trip_id, flight.dep_key) )
                flights_per_passenger[str(passenger.trip_id)]["one_one"]["values"].append(var_cost)
                flights_per_passenger[str(passenger.trip_id)]["one_one"]["pax_cnt"].append(passenger.pax_cnt)
                flights_per_passenger[str(passenger.trip_id)]["one_one"]["avail_cnt"].append(flight.c_avail_cnt + flight.y_avail_cnt)

                # We add the assigned flight to be dictionary saving the global variables
                passengers_per_flight[flight.dep_key]["one_one"].append( ((passenger.trip_id, flight.dep_key), passenger.pax_cnt) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_one_one = n_one_one + 1

## Forced one case

In [107]:
for passenger in pnr_list:
    print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))
    
    if (passenger.booked_multi_leg):
        available_flights_case = flights_out_in_dic[ (str(passenger.oper_od_orig_cd), str(passenger.oper_od_dest_cd)) ]
        
        # Now we iterate over all possible case
        for flight in available_flights_case:

            # Retrieve the new time departures
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 
            
            # Retrieve the original time departures
            orig_time_dep = datetime.strptime(passenger.ideal_dep_dtmz, "%Y-%m-%d %H:%M")
            orig_time_arr = datetime.strptime(passenger.ideal_arr_dtmz, "%Y-%m-%d %H:%M")
                
            time_diff_dep = ((new_time_dep - orig_time_dep).total_seconds() / 60.0) # Minutes
            time_diff_arr = ((new_time_arr - orig_time_arr).total_seconds() / 60.0) 

            cost_dep = time_penalty(time_diff_dep)
            cost_arr = time_penalty(time_diff_arr)

            # Is this a good pairing?
            if ((cost_dep <= 0) or (cost_arr <= 0)): # Check the penalty
                #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:
                var_cost = cost_dep + cost_arr + cvm_factor["forced_one"]*passenger.cvm
                
                # We save the variable

                # We save the assigned DIRECT flight for this passenger
                #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )
                flights_per_passenger[str(passenger.trip_id)]["forced_one"]["variables"].append((passenger.trip_id, flight.dep_key) )
                flights_per_passenger[str(passenger.trip_id)]["forced_one"]["values"].append(var_cost)
                flights_per_passenger[str(passenger.trip_id)]["forced_one"]["pax_cnt"].append(passenger.pax_cnt)
                flights_per_passenger[str(passenger.trip_id)]["forced_one"]["avail_cnt"].append(flight.c_avail_cnt + flight.y_avail_cnt)

                # We add the assigned flight to be dictionary saving the global variables
                passengers_per_flight[flight.dep_key]["forced_one"].append( ((passenger.trip_id, flight.dep_key), passenger.pax_cnt) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_forced_one = n_forced_one + 1

(Passenger, trip number):  (0, 1) / 15096
(Passenger, trip number):  (1, 1) / 15096
(Passenger, trip number):  (2, 1) / 15096
(Passenger, trip number):  (3, 2) / 15096
(Passenger, trip number):  (4, 1) / 15096
(Passenger, trip number):  (5, 2) / 15096
(Passenger, trip number):  (6, 2) / 15096
(Passenger, trip number):  (7, 2) / 15096
(Passenger, trip number):  (8, 2) / 15096
(Passenger, trip number):  (9, 1) / 15096
(Passenger, trip number):  (10, 1) / 15096
(Passenger, trip number):  (11, 1) / 15096
(Passenger, trip number):  (12, 2) / 15096
(Passenger, trip number):  (13, 1) / 15096
(Passenger, trip number):  (14, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (16, 1) / 15096
(Passenger, trip number):  (17, 1) / 15096
(Passenger, trip number):  (18, 1) / 15096
(Passenger, trip number):  (19, 1) / 15096
(Passenger, trip number):  (20, 1) / 15096
(Passenger, trip number):  (21, 1) / 15096
(Passenger, trip numb

## Multi-one case

In [108]:
for passenger in pnr_list:
    print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))
    
    if (passenger.booked_multi_leg):
        available_flights_case = flights_out_in_dic[ (str(passenger.orig_cd), str(passenger.dest_cd)) ]
        
        # Now we iterate over all possible case
        for flight in available_flights_case:

            # Retrieve the new time departures
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 
            
            # Retrieve the original time departures
            orig_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
            orig_time_arr = datetime.strptime(passenger.arr_dtmz, "%Y-%m-%d %H:%M")
                
            time_diff_dep = ((new_time_dep - orig_time_dep).total_seconds() / 60.0) # Minutes
            time_diff_arr = ((new_time_arr - orig_time_arr).total_seconds() / 60.0) 

            cost_dep = time_penalty(time_diff_dep)
            cost_arr = time_penalty(time_diff_arr)

            # Is this a good pairing?
            if ((cost_dep <= 0) or (cost_arr <= 0)): # Check the penalty
                #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:
                var_cost = cost_dep + cost_arr + cvm_factor["multi_one"]*passenger.cvm
                
                # We save the variable

                # We save the assigned DIRECT flight for this passenger
                #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )
                flights_per_passenger[str(passenger.trip_id)]["multi_one"]["variables"].append((passenger.trip_id, flight.dep_key) )
                flights_per_passenger[str(passenger.trip_id)]["multi_one"]["values"].append(var_cost)
                flights_per_passenger[str(passenger.trip_id)]["multi_one"]["pax_cnt"].append(passenger.pax_cnt)
                flights_per_passenger[str(passenger.trip_id)]["multi_one"]["avail_cnt"].append(flight.c_avail_cnt + flight.y_avail_cnt)
                

                # We add the assigned flight to be dictionary saving the global variables
                passengers_per_flight[flight.dep_key]["multi_one"].append( ((passenger.trip_id, flight.dep_key), passenger.pax_cnt) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_multi_one = n_multi_one + 1

(Passenger, trip number):  (0, 1) / 15096
(Passenger, trip number):  (1, 1) / 15096
(Passenger, trip number):  (2, 1) / 15096
(Passenger, trip number):  (3, 2) / 15096
(Passenger, trip number):  (4, 1) / 15096
(Passenger, trip number):  (5, 2) / 15096
(Passenger, trip number):  (6, 2) / 15096
(Passenger, trip number):  (7, 2) / 15096
(Passenger, trip number):  (8, 2) / 15096
(Passenger, trip number):  (9, 1) / 15096
(Passenger, trip number):  (10, 1) / 15096
(Passenger, trip number):  (11, 1) / 15096
(Passenger, trip number):  (12, 2) / 15096
(Passenger, trip number):  (13, 1) / 15096
(Passenger, trip number):  (14, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (16, 1) / 15096
(Passenger, trip number):  (17, 1) / 15096
(Passenger, trip number):  (18, 1) / 15096
(Passenger, trip number):  (19, 1) / 15096
(Passenger, trip number):  (20, 1) / 15096
(Passenger, trip number):  (21, 1) / 15096
(Passenger, trip numb

In [109]:
passengers_per_flight['AZ20271023RHPTPH17919']

{'one_one': [(('387_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 5),
  (('639_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('877_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('922_2_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('985_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 3),
  (('2111_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('2999_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('3677_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 2),
  (('5941_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 4),
  (('6335_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 4),
  (('8761_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('8846_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 4),
  (('9103_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('9627_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 4),
  (('9728_1_AZ20271021RHPTPH851', 'AZ20271023RHPTPH17919'), 1),
  (('10646_1_AZ20271021RHPTPH851',

## One-Multi

In [110]:
for passenger in pnr_list:
    print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))
    
    if (not passenger.booked_multi_leg):
        
        #available_flights_case = flights_2legs_out_in_dic[(str(passenger.oper_od_orig_cd), str(passenger.oper_od_dest_cd))]
        available_flights_case = flights_2legs_out_in_dic[(str(passenger.orig_cd), str(passenger.dest_cd))]
        # Now we iterate over the flights
        for flight_combination in available_flights_case:
            
            first_leg = flight_combination[0]
            second_leg = flight_combination[1]
            
            # Check the departure penalty costs FIRST before doiny any computation
            original_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_dep = datetime.strptime(first_leg.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            time_diff_dep = ((new_time_dep - original_time_dep).total_seconds() / 60.0)  # in minutes
                        
            # Check if the penalty is acceptable or not
            cost_dep = time_penalty(time_diff_dep)
            if (cost_dep <= 0): # Check the penalty
                # Skip all the pairings with ths flight as the first leg
                continue

            # Now we check if the second flight is trash in terms of arriving
            original_time_arr = datetime.strptime(passenger.arr_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_arr = datetime.strptime(second_leg.arr_dtmz, "%Y-%m-%d %H:%M")
            time_diff_arr = ((new_time_arr - original_time_arr).total_seconds() / 60.0)  # in minutes

            # Check if the cost is acceptable or not
            cost_arr = time_penalty(time_diff_arr)
            if (cost_arr <= 0):
                # Skip this second leg
                continue
            
            # Now we add the variable!
            var = (passenger.trip_id, (first_leg.dep_key, second_leg.dep_key))
            
            var_cost = cost_dep + cost_arr + cvm_factor["one_multi"]*passenger.cvm        
                
            # We save the variable

            # We save the assigned DIRECT flight for this passenger
            #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )
            flights_per_passenger[str(passenger.trip_id)]["one_multi"]["variables"].append( var )
            flights_per_passenger[str(passenger.trip_id)]["one_multi"]["values"].append(var_cost)
            flights_per_passenger[str(passenger.trip_id)]["one_multi"]["pax_cnt"].append(passenger.pax_cnt)
            flights_per_passenger[str(passenger.trip_id)]["one_multi"]["avail_cnt"].append((first_leg.c_avail_cnt + first_leg.y_avail_cnt, second_leg.c_avail_cnt + second_leg.y_avail_cnt))
            

            # We add the assigned flight to be dictionary saving the global variables
            passengers_per_flight[first_leg.dep_key]["one_multi"].append( (var, passenger.pax_cnt) )
            passengers_per_flight[second_leg.dep_key]["one_multi"].append( (var, passenger.pax_cnt) )



            # We increment the number of total variables
            n_variables = n_variables + 1
            n_one_multi = n_one_multi + 1

(Passenger, trip number):  (0, 1) / 15096
(Passenger, trip number):  (1, 1) / 15096
(Passenger, trip number):  (2, 1) / 15096
(Passenger, trip number):  (3, 2) / 15096
(Passenger, trip number):  (4, 1) / 15096
(Passenger, trip number):  (5, 2) / 15096
(Passenger, trip number):  (6, 2) / 15096
(Passenger, trip number):  (7, 2) / 15096
(Passenger, trip number):  (8, 2) / 15096
(Passenger, trip number):  (9, 1) / 15096
(Passenger, trip number):  (10, 1) / 15096
(Passenger, trip number):  (11, 1) / 15096
(Passenger, trip number):  (12, 2) / 15096
(Passenger, trip number):  (13, 1) / 15096
(Passenger, trip number):  (14, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (16, 1) / 15096
(Passenger, trip number):  (17, 1) / 15096
(Passenger, trip number):  (18, 1) / 15096
(Passenger, trip number):  (19, 1) / 15096
(Passenger, trip number):  (20, 1) / 15096
(Passenger, trip number):  (21, 1) / 15096
(Passenger, trip numb

## Multi-Multi

In [111]:
for passenger in pnr_list:
    print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))
    
    if (passenger.booked_multi_leg):
        
        #available_flights_case = flights_2legs_out_in_dic[(str(passenger.oper_od_orig_cd), str(passenger.oper_od_dest_cd))]
        available_flights_case = flights_2legs_out_in_dic[(str(passenger.orig_cd), str(passenger.dest_cd))]
        # Now we iterate over the flights
        for flight_combination in available_flights_case:
            
            first_leg = flight_combination[0]
            second_leg = flight_combination[1]
            
            # Check the departure penalty costs FIRST before doiny any computation
            original_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_dep = datetime.strptime(first_leg.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            time_diff_dep = ((new_time_dep - original_time_dep).total_seconds() / 60.0)  # in minutes
                        
            # Check if the penalty is acceptable or not
            cost_dep = time_penalty(time_diff_dep)
            if (cost_dep <= 0): # Check the penalty
                # Skip all the pairings with ths flight as the first leg
                continue

            # Now we check if the second flight is trash in terms of arriving
            original_time_arr = datetime.strptime(passenger.arr_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_arr = datetime.strptime(second_leg.arr_dtmz, "%Y-%m-%d %H:%M")
            time_diff_arr = ((new_time_arr - original_time_arr).total_seconds() / 60.0)  # in minutes

            # Check if the cost is acceptable or not
            cost_arr = time_penalty(time_diff_arr)
            if (cost_arr <= 0):
                # Skip this second leg
                continue
            
            # Now we add the variable!
            var = (passenger.trip_id, (first_leg.dep_key, second_leg.dep_key))
            
            var_cost = cost_dep + cost_arr + cvm_factor["multi_multi"]*passenger.cvm        
                
            # We save the variable

            # We save the assigned DIRECT flight for this passenger
            #variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )
            flights_per_passenger[str(passenger.trip_id)]["multi_multi"]["variables"].append( var )
            flights_per_passenger[str(passenger.trip_id)]["multi_multi"]["values"].append(var_cost)
            flights_per_passenger[str(passenger.trip_id)]["multi_multi"]["pax_cnt"].append(passenger.pax_cnt)
            flights_per_passenger[str(passenger.trip_id)]["one_multi"]["avail_cnt"].append((first_leg.c_avail_cnt + first_leg.y_avail_cnt, second_leg.c_avail_cnt + second_leg.y_avail_cnt))
            

            # We add the assigned flight to be dictionary saving the global variables
            passengers_per_flight[first_leg.dep_key]["multi_multi"].append( (var, passenger.pax_cnt) )
            passengers_per_flight[second_leg.dep_key]["multi_multi"].append( (var, passenger.pax_cnt) )



            # We increment the number of total variables
            n_variables = n_variables + 1
            n_multi_multi = n_multi_multi + 1

(Passenger, trip number):  (0, 1) / 15096
(Passenger, trip number):  (1, 1) / 15096
(Passenger, trip number):  (2, 1) / 15096
(Passenger, trip number):  (3, 2) / 15096
(Passenger, trip number):  (4, 1) / 15096
(Passenger, trip number):  (5, 2) / 15096
(Passenger, trip number):  (6, 2) / 15096
(Passenger, trip number):  (7, 2) / 15096
(Passenger, trip number):  (8, 2) / 15096
(Passenger, trip number):  (9, 1) / 15096
(Passenger, trip number):  (10, 1) / 15096
(Passenger, trip number):  (11, 1) / 15096
(Passenger, trip number):  (12, 2) / 15096
(Passenger, trip number):  (13, 1) / 15096
(Passenger, trip number):  (14, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (15, 1) / 15096
(Passenger, trip number):  (16, 1) / 15096
(Passenger, trip number):  (17, 1) / 15096
(Passenger, trip number):  (18, 1) / 15096
(Passenger, trip number):  (19, 1) / 15096
(Passenger, trip number):  (20, 1) / 15096
(Passenger, trip number):  (21, 1) / 15096
(Passenger, trip numb

In [112]:
flights_per_passenger[ list(flights_per_passenger.keys())[2] ]

{'one_one': {'variables': [], 'values': [], 'pax_cnt': [], 'avail_cnt': []},
 'forced_one': {'variables': [], 'values': [], 'pax_cnt': [], 'avail_cnt': []},
 'multi_one': {'variables': [('2_1_AZ20271021TPHOLT17144',
    'AZ20271022TPHOLT8798'),
   ('2_1_AZ20271021TPHOLT17144', 'AZ20271023TPHOLT8798'),
   ('2_1_AZ20271021TPHOLT17144', 'AZ20271022TPHOLT17144'),
   ('2_1_AZ20271021TPHOLT17144', 'AZ20271023TPHOLT17144'),
   ('2_1_AZ20271021TPHOLT17144', 'AZ20271022TPHOLT5535')],
  'values': [197.114198, 177.114198, 197.114198, 177.114198, 187.114198],
  'pax_cnt': [1, 1, 1, 1, 1],
  'avail_cnt': [1, -3, 2, -3, -2]},
 'one_multi': {'variables': [], 'values': [], 'pax_cnt': [], 'avail_cnt': []},
 'multi_multi': {'variables': [],
  'values': [],
  'pax_cnt': [],
  'avail_cnt': []}}

In [113]:
n_variables

125452

In [114]:
t2 = time.time()
print("Variables created! ")
print("Seconds elapsed: ", t2 - t1)

Variables created! 
Seconds elapsed:  134.14127469062805


# Craetion of variables to optimize the model!

In [115]:
## Now we create the dictionaries! 
# For local constraints
variables_per_passenger = {}
# For seat constraints!
variables_per_flight = {}

In [116]:
# We initialize the dictionaries
for passenger in list( flights_per_passenger.keys() ):
    variables_per_passenger[passenger] = []

for idx in range(len(available_flights)):
    variables_per_flight[available_flights[idx].dep_key] = {}
    available_seats = available_flights[idx].c_avail_cnt + available_flights[idx].y_avail_cnt
    #
    # Now we save the variables
    variables_per_flight[available_flights[idx].dep_key]["variables"] = []
    variables_per_flight[available_flights[idx].dep_key]["pax_cnt"] = []
    #variables_per_flight[available_flights[idx].dep_key]["avail_cnt"] = available_seats
    variables_per_flight[available_flights[idx].dep_key]["avail_cnt"] = []


# Now we create the variables ! 

In [117]:
#flights_per_passenger

# CREATION OF THE MODEL !!! 

In [118]:
from dimod import ConstrainedQuadraticModel
from dwave.system import LeapHybridCQMSampler

In [119]:
#! pip install multiverse_singularity_optimization-1.9.1-py3-none-any.whl 
import singularity.optimization as sop

In [120]:
ancilla_variable = sop.Variable("ancilla")
# We initialize the total cost
Total_cost = 0*ancilla_variable 

In [122]:
n = 0
for passenger in list( flights_per_passenger.keys() ):
    print("Cost: ", n, " / ", len(flights_per_passenger.keys()))
    mini_dict = flights_per_passenger[passenger]
    single_variables = mini_dict["one_one"]["variables"] + mini_dict["forced_one"]["variables"] + mini_dict["multi_one"]["variables"]
    single_values = mini_dict["one_one"]["values"] + mini_dict["forced_one"]["values"] + mini_dict["multi_one"]["values"]
    single_pax_cnt = mini_dict["one_one"]["pax_cnt"] + mini_dict["forced_one"]["pax_cnt"] + mini_dict["multi_one"]["pax_cnt"]
    single_avail_cnt = mini_dict["one_one"]["avail_cnt"] + mini_dict["forced_one"]["avail_cnt"] + mini_dict["multi_one"]["avail_cnt"]

    double_variables = mini_dict["one_multi"]["variables"] + mini_dict["multi_multi"]["variables"]
    double_values = mini_dict["one_multi"]["values"] + mini_dict["multi_multi"]["values"]
    double_pax_cnt = mini_dict["one_multi"]["pax_cnt"] + mini_dict["multi_multi"]["pax_cnt"]
    double_avail_cnt = mini_dict["one_multi"]["avail_cnt"] + mini_dict["multi_multi"]["avail_cnt"]

    for idx in range(len(single_variables)):
        if (len(single_variables) != 0):
            flight_key = single_variables[idx][1]
            
            # The whole variable is the passenger id PLUS the flight
            single_variable = sop.Variable(str(single_variables[idx]))
            single_value = single_values[idx]
            Total_cost = Total_cost + single_variable * single_value

            # We save the variables ! 
            variables_per_passenger[passenger].append(single_variable)

            # For the airplanes! 
            variables_per_flight[flight_key]["variables"].append(single_variable)
            variables_per_flight[flight_key]["pax_cnt"].append(single_pax_cnt[idx])
            if single_avail_cnt[idx] <= 0:
                variables_per_flight[flight_key]["avail_cnt"].append(0)
            else:
                variables_per_flight[flight_key]["avail_cnt"].append(single_avail_cnt[idx])

    for idx2 in range(len(double_variables)):
        if (len(double_variables) != 0):
            flight1, flight2 = double_variables[idx2][1][0], double_variables[idx2][1][1]

            # The whole variable is the apssenger id PLUS the flights
            double_variable = sop.Variable(str(double_variables[idx2]))
            double_value = double_values[idx2]
            Total_cost = Total_cost + double_variable * double_value

            # We save the variables
            variables_per_passenger[passenger].append(double_variable)

            # For the airplanes! 
            ## First plane
            variables_per_flight[flight1]["variables"].append(double_variable)
            variables_per_flight[flight1]["pax_cnt"].append(double_pax_cnt[idx2])
            if double_avail_cnt[idx2][0] <= 0:
                variables_per_flight[flight1]["avail_cnt"].append(0)
            else:
                variables_per_flight[flight1]["avail_cnt"].append(double_avail_cnt[idx2][0])

            ## Second plane
            variables_per_flight[flight2]["variables"].append(double_variable)
            variables_per_flight[flight2]["pax_cnt"].append(double_pax_cnt[idx2])
            if double_avail_cnt[idx2][1] <= 0:
                variables_per_flight[flight2]["avail_cnt"].append(0)
            else:
                variables_per_flight[flight2]["avail_cnt"].append(double_avail_cnt[idx2][1])

    n = n + 1


    

Constraint:  0  /  15096
Constraint:  1  /  15096
Constraint:  2  /  15096
Constraint:  3  /  15096
Constraint:  4  /  15096
Constraint:  5  /  15096
Constraint:  6  /  15096
Constraint:  7  /  15096
Constraint:  8  /  15096
Constraint:  9  /  15096
Constraint:  10  /  15096
Constraint:  11  /  15096
Constraint:  12  /  15096
Constraint:  13  /  15096
Constraint:  14  /  15096
Constraint:  15  /  15096
Constraint:  16  /  15096
Constraint:  17  /  15096
Constraint:  18  /  15096
Constraint:  19  /  15096
Constraint:  20  /  15096
Constraint:  21  /  15096
Constraint:  22  /  15096
Constraint:  23  /  15096
Constraint:  24  /  15096
Constraint:  25  /  15096
Constraint:  26  /  15096
Constraint:  27  /  15096
Constraint:  28  /  15096
Constraint:  29  /  15096
Constraint:  30  /  15096
Constraint:  31  /  15096
Constraint:  32  /  15096
Constraint:  33  /  15096
Constraint:  34  /  15096
Constraint:  35  /  15096
Constraint:  36  /  15096
Constraint:  37  /  15096
Constraint:  38  /  15

In [123]:
variables_per_flight.keys()

dict_keys(['AZ20271023RHPTPH17919', 'AZ20271021RHPTPH17919', 'AZ20271024RHPTPH17919', 'AZ20271025RHPTPH17919', 'AZ20271027RHPTPH17919', 'AZ20271020RHPTPH17919', 'AZ20271028RHPTPH17919', 'AZ20271022JDPTPH19123', 'AZ20271024JDPTPH19123', 'AZ20271021JDPTPH19123', 'AZ20271023JDPTPH19123', 'AZ20271020JDPTPH19123', 'AZ20271027JDPTPH19123', 'AZ20271026JDPTPH19123', 'AZ20271028JDPTPH19123', 'AZ20271025JDPTPH19123', 'AZ20271019JDPTPH19123', 'AZ20271023TPHNWY10061', 'AZ20271028TPHNWY10061', 'AZ20271024TPHNWY10061', 'AZ20271027TPHNWY10061', 'AZ20271025TPHNWY10061', 'AZ20271020TPHNWY10061', 'AZ20271022TPHNWY10061', 'AZ20271019TPHNWY10061', 'AZ20271026TPHNWY10061', 'AZ20271028TPHJDP6704', 'AZ20271025TPHJDP6704', 'AZ20271027TPHJDP6704', 'AZ20271023TPHJDP6704', 'AZ20271026TPHJDP6704', 'AZ20271021TPHJDP6704', 'AZ20271024TPHJDP6704', 'AZ20271022TPHJDP6704', 'AZ20271020ECZTPH10704', 'AZ20271024ECZTPH10704', 'AZ20271022ECZTPH10704', 'AZ20271027ECZTPH10704', 'AZ20271024VCFTPH17724', 'AZ20271019VCFTPH17724

In [124]:
variables_per_flight["AZ20271022TPHPJY16397"]

{'variables': [('205_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('269_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('274_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('520_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('647_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1185_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1196_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1416_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1762_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1891_3_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1899_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1946_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('1975_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('2005_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('2404_2_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('2475_1_AZ20271020TPHPJY16397', 'AZ20271022TPHPJY16397'),
  ('2591_2_AZ202

In [125]:
#variables_per_passenger["1772_1_AZ20271021TPHRBR5443"]

# Now we add the seat constraints

In [126]:
seat_constraints = []

for flight in list( variables_per_flight.keys() ):

    sub_dict = variables_per_flight[flight]

    # Some flights are not considered, SKIP !!!
    if all(not v for v in sub_dict.values()):
        print("Skipping flight: ", flight)
        continue  # Skip this flight

    passenger_count = sub_dict["pax_cnt"]
    vars = sub_dict["variables"]
    avail_seats = sub_dict["avail_cnt"][0]

    # Now we perform the weighted sum
    sum_passengers = sum( np.array(vars) * np.array(passenger_count) )

    constraint = sop.Constraint(lhs = sum_passengers, operator="<=", 
                               rhs = avail_seats, penalty_strength= int(1e8),
                               name = f"Seats_{flight}")
    seat_constraints.append(constraint)

Skipping flight:  AZ20271024RHPTPH17919
Skipping flight:  AZ20271025RHPTPH17919
Skipping flight:  AZ20271027RHPTPH17919
Skipping flight:  AZ20271020RHPTPH17919
Skipping flight:  AZ20271028RHPTPH17919
Skipping flight:  AZ20271024JDPTPH19123
Skipping flight:  AZ20271020JDPTPH19123
Skipping flight:  AZ20271027JDPTPH19123
Skipping flight:  AZ20271026JDPTPH19123
Skipping flight:  AZ20271028JDPTPH19123
Skipping flight:  AZ20271025JDPTPH19123
Skipping flight:  AZ20271019JDPTPH19123
Skipping flight:  AZ20271028TPHNWY10061
Skipping flight:  AZ20271024TPHNWY10061
Skipping flight:  AZ20271027TPHNWY10061
Skipping flight:  AZ20271025TPHNWY10061
Skipping flight:  AZ20271020TPHNWY10061
Skipping flight:  AZ20271019TPHNWY10061
Skipping flight:  AZ20271026TPHNWY10061
Skipping flight:  AZ20271028TPHJDP6704
Skipping flight:  AZ20271025TPHJDP6704
Skipping flight:  AZ20271027TPHJDP6704
Skipping flight:  AZ20271026TPHJDP6704
Skipping flight:  AZ20271024TPHJDP6704
Skipping flight:  AZ20271020ECZTPH10704
Skipp

In [127]:
print("These flights were not even considered! Example: ")
variables_per_flight["AZ20271024JDPTPH19123"]

These flights were not even considered! Example: 


{'variables': [], 'pax_cnt': [], 'avail_cnt': []}

# Now we add the variable constraints

In [128]:
# Number of reaccommodations to assign to each passenger
min_reaccommodations = 1
passenger_constraints = []

for passenger in list( variables_per_passenger.keys() ):
    sum_constraint_passenger = sum(variables_per_passenger[passenger])

    constraint = sop.Constraint(lhs = sum_constraint_passenger, operator=">=", 
                               rhs = min_reaccommodations, penalty_strength= int(1e5),
                               name = f"Min_reaccommodations_{passenger}")
    passenger_constraints.append(constraint)

In [129]:
# Number of reaccommodations to assign to each passenger
max_reaccommodations = 5

for passenger in list( variables_per_passenger.keys() ):
    sum_constraint_passenger = sum(variables_per_passenger[passenger])

    constraint = sop.Constraint(lhs = sum_constraint_passenger, operator="<=", 
                               rhs = max_reaccommodations, penalty_strength= int(1e3),
                               name = f"Max_reaccommodations_{passenger}")
    passenger_constraints.append(constraint)

# Now we obtain the model !!! 

In [130]:
objective = sop.Objective(Total_cost, "maximize")
Reaccommodation = sop.Model(objective)

## Now we add the constraints

In [131]:
total_constraints = seat_constraints + passenger_constraints
for idx in range(len(total_constraints)):
    print("Adding constraint: ", idx, "/", len(total_constraints))
    Reaccommodation.add_constraint(total_constraints[idx])

Adding constraint:  0 / 30960
Adding constraint:  1 / 30960
Adding constraint:  2 / 30960
Adding constraint:  3 / 30960
Adding constraint:  4 / 30960
Adding constraint:  5 / 30960
Adding constraint:  6 / 30960
Adding constraint:  7 / 30960
Adding constraint:  8 / 30960
Adding constraint:  9 / 30960
Adding constraint:  10 / 30960
Adding constraint:  11 / 30960
Adding constraint:  12 / 30960
Adding constraint:  13 / 30960
Adding constraint:  14 / 30960
Adding constraint:  15 / 30960
Adding constraint:  16 / 30960
Adding constraint:  17 / 30960
Adding constraint:  18 / 30960
Adding constraint:  19 / 30960
Adding constraint:  20 / 30960
Adding constraint:  21 / 30960
Adding constraint:  22 / 30960
Adding constraint:  23 / 30960
Adding constraint:  24 / 30960
Adding constraint:  25 / 30960
Adding constraint:  26 / 30960
Adding constraint:  27 / 30960
Adding constraint:  28 / 30960
Adding constraint:  29 / 30960
Adding constraint:  30 / 30960
Adding constraint:  31 / 30960
Adding constraint:

KeyboardInterrupt: 

In [ ]:
len(passenger_constraints)

In [ ]:
len(pnr_list)

In [ ]:
print("Total number of variables", n_one_one + n_one_multi + n_forced_one + n_multi_one + n_multi_multi)

# Now we optimize ! 

In [ ]:
# This one supports up to 1M variables!

#solver = "dwave_hybrid"
#solver = "simulated_annealing"
solver = "classical"

# Import solver ! 
#import neal
#solver=neal.SimulatedAnnealingSampler()

In [ ]:
time_in = time.time()

In [ ]:
result = Reaccommodation.optimize(solver="classical")
if solver == "classical":
    result = Reaccommodation.optimize(solver=solver)
else:
    result = Reaccommodation.optimize(solver= solver, num_solutions = 10, 
                                  time_limit= 60*5, dwave_api_token = DWAVE_API_TOKEN
                                  )


In [ ]:
time_out = time.time()
print("Minutes taken to optimize: ", (time_out - time_in) / 60)

with open("time_taken_DWAVE.txt", "w") as f:
    f.write(str((time_in - time_out)/60) )

In [ ]:
print("Best solution: ")
print("Objective value: ", result.objective_value)
BestResults = result.values
BestResults

# Nowe we create a dictionary where to save the results

In [ ]:
ReaccommodationResults = {}

for idx in range(len(pnr_list)):
    parts = pnr_list[idx].trip_id
    parts = parts.split("_")

    new_key = f"{parts[0]}_{parts[2]}"

    # Now we save the new keys
    ReaccommodationResults[new_key] = []

In [ ]:
#ReaccommodationResults

# Now we save the results

In [ ]:
import re
import ast

# Updated regex pattern to handle nested tuples
pattern = r"\('(\d+)_(\d+)_([^']+)',\s*(\(.+\)|'[^']+')\)"

In [ ]:
for key, value in BestResults.items():

    # We skip this option if it is not correct ! 
    if value != 1:
        continue
    # We obtain the name of the variable
    item = key.name
    
    # We check the match
    match = re.match(pattern, item)
    
    if match:
        pnr_recloc, trip_number, flight_dep_key, flight_reaccommodations = match.groups()
        #print("pnr_recloc:", pnr_recloc, "flight_dep_key:", flight_dep_key, "flight_reaccommodations:", flight_reaccommodations)

        new_key = f"{pnr_recloc}_{flight_dep_key}"

        # We save the results ! 
        ReaccommodationResults[new_key].append( ast.literal_eval(flight_reaccommodations) )

    else:
        print("No match for:", item)
    



    #ancilla_str = str(key).split(",")[0][2:-1].split("_")
    #recloc_nr, flight_nr = ancilla_str[0], ancilla_str[2]

In [ ]:
list(ReaccommodationResults.values())[0]

# Now we obtain the reaccommodation results 

In [ ]:
n_reaccommodated = 0

for reaccommodation in ReaccommodationResults.values():
    if reaccommodation != []:
        n_reaccommodated = n_reaccommodated + 1

print("Percentage of reaccommodation: ",  n_reaccommodated / len(pnr_list) * 100)

with open("Reaccommodation_percentage.txt", "w") as f:
    f.write(str( n_reaccommodated / len(pnr_list) * 100)  )

In [ ]:
print(ReaccommodationResults)

# Now we save the results in the desired format 

In [ ]:
RECLOC = []
CABIN_CD = []
COS_CD = []
OPER_OD_ORIG_CD = []
OPER_OD_DEST_CD = []
DEP_KEY = []
DEP_DT = []
ORIG_CD = []
DEST_CD = []
FLT_NUM = []
DEP_DTML = []
ARR_DTML = []
DEP_DTMZ = []
ARR_DTMZ = []
PREV_OD_BROKEN_IND = []
PAX_CNT = []
CVM = []
PREV_CONN_TIME = []

# New arrays
ALT_CABIN_CD = []
ALT_OPER_OD_ORIG_CD = []
ALT_OPER_OD_DEST_CD = []
ALT_DEP_DT = []
ALT_ORIG_CD = []
ALT_DEST_CD = []
ALT_FLT_NUM = []
ALT_DEP_DTML = []
ALT_ARR_DTML = []
ALT_DEP_DTMZ = []
ALT_ARR_DTMZ = []
ALT_CONN_TIME_MINS = []

In [ ]:
for key, values in ReaccommodationResults.items():
    given_pnr = key.split("_")[0]
    given_dep_key = key.split("_")[1]

    #given_pnr_df = pnr_df[(pnr_df["RECLOC"] == given_pnr) & (pnr_df["DEP_KEY"] == given_dep_key)]
    given_pnr_df = pnr_df[ (pnr_df["RECLOC"] == int(given_pnr)) & (pnr_df["DEP_KEY"] == given_dep_key) ]
    # Now we iterate over the results 

    # NO SOLUTIONS !!!! 
    if values == []:
        print("RECLOC, FLIGHT Has no options: ", given_pnr, ", ", given_dep_key)
        RECLOC.append(given_pnr_df["RECLOC"].values[0])
        CABIN_CD.append(given_pnr_df["CABIN_CD"].values[0])
        COS_CD.append(given_pnr_df["COS_CD"].values[0])
        OPER_OD_ORIG_CD.append(given_pnr_df["OPER_OD_ORIG_CD"].values[0])
        OPER_OD_DEST_CD.append(given_pnr_df["OPER_OD_DEST_CD"].values[0])
        DEP_KEY.append(given_pnr_df["DEP_KEY"].values[0])
        DEP_DT.append(given_pnr_df["DEP_DT"].values[0])
        ORIG_CD.append(given_pnr_df["ORIG_CD"].values[0])
        DEST_CD.append(given_pnr_df["DEST_CD"].values[0])
        FLT_NUM.append(given_pnr_df["FLT_NUM"].values[0])
        DEP_DTML.append(given_pnr_df["DEP_DTML"].values[0])
        ARR_DTML.append(given_pnr_df["ARR_DTML"].values[0])
        DEP_DTMZ.append(given_pnr_df["DEP_DTMZ"].values[0])
        ARR_DTMZ.append(given_pnr_df["ARR_DTMZ"].values[0])
        PREV_OD_BROKEN_IND.append(given_pnr_df["OD_BROKEN_IND"].values[0])
        PAX_CNT.append(given_pnr_df["PAX_CNT"].values[0])
        CVM.append(given_pnr_df["CVM"].values[0])
        PREV_CONN_TIME.append(given_pnr_df["CONN_TIME_MINS"].values[0])

        # Now for the reaaccommodation
        ALT_CABIN_CD.append(None)
        ALT_OPER_OD_ORIG_CD.append(None)
        ALT_OPER_OD_DEST_CD.append(None)
        ALT_DEP_DT.append(None)
        ALT_ORIG_CD.append(None)
        ALT_DEST_CD.append(None)
        ALT_FLT_NUM.append(None)
        ALT_DEP_DTML.append(None)
        ALT_ARR_DTML.append(None)
        ALT_DEP_DTMZ.append(None)
        ALT_ARR_DTMZ.append(None)
        ALT_CONN_TIME_MINS.append(None)


    else: 
        for reaccommodation in values:
            
            # Case where we have a single alternate flight ! 
            if type(reaccommodation) == str:
                given_avail_flight_df = available_flights_df[available_flights_df["DEP_KEY"] == reaccommodation]

                RECLOC.append(given_pnr_df["RECLOC"].values[0])
                CABIN_CD.append(given_pnr_df["CABIN_CD"].values[0])
                COS_CD.append(given_pnr_df["COS_CD"].values[0])
                OPER_OD_ORIG_CD.append(given_pnr_df["OPER_OD_ORIG_CD"].values[0])
                OPER_OD_DEST_CD.append(given_pnr_df["OPER_OD_DEST_CD"].values[0])
                DEP_KEY.append(given_pnr_df["DEP_KEY"].values[0])
                DEP_DT.append(given_pnr_df["DEP_DT"].values[0])
                ORIG_CD.append(given_pnr_df["ORIG_CD"].values[0])
                DEST_CD.append(given_pnr_df["DEST_CD"].values[0])
                FLT_NUM.append(given_pnr_df["FLT_NUM"].values[0])
                DEP_DTML.append(given_pnr_df["DEP_DTML"].values[0])
                ARR_DTML.append(given_pnr_df["ARR_DTML"].values[0])
                DEP_DTMZ.append(given_pnr_df["DEP_DTMZ"].values[0])
                ARR_DTMZ.append(given_pnr_df["ARR_DTMZ"].values[0])
                PREV_OD_BROKEN_IND.append(given_pnr_df["OD_BROKEN_IND"].values[0])
                PAX_CNT.append(given_pnr_df["PAX_CNT"].values[0])
                CVM.append(given_pnr_df["CVM"].values[0])
                PREV_CONN_TIME.append(given_pnr_df["CONN_TIME_MINS"].values[0])

                # Now for the reaaccommodation
                ALT_CABIN_CD.append("Either")
                ALT_OPER_OD_ORIG_CD.append(given_avail_flight_df["ORIG_CD"].values[0])
                ALT_OPER_OD_DEST_CD.append(given_avail_flight_df["DEST_CD"].values[0])
                ALT_DEP_DT.append(given_avail_flight_df["DEP_DT"].values[0])
                ALT_ORIG_CD.append(given_avail_flight_df["ORIG_CD"].values[0])
                ALT_DEST_CD.append(given_avail_flight_df["DEST_CD"].values[0])
                ALT_FLT_NUM.append(given_avail_flight_df["FLT_NUM"].values[0])
                ALT_DEP_DTML.append(given_avail_flight_df["DEP_DTML"].values[0])
                ALT_ARR_DTML.append(given_avail_flight_df["ARR_DTML"].values[0])
                ALT_DEP_DTMZ.append(given_avail_flight_df["DEP_DTMZ"].values[0])
                ALT_ARR_DTMZ.append(given_avail_flight_df["ARR_DTMZ"].values[0])
                ALT_CONN_TIME_MINS.append( None )
            
            # Two reaccommodations ! 
            elif type(reaccommodation) == tuple:
                for leg in reaccommodation:
                    leg_df = available_flights_df[available_flights_df["DEP_KEY"] == leg]

                    RECLOC.append(given_pnr_df["RECLOC"].values[0])
                    CABIN_CD.append(given_pnr_df["CABIN_CD"].values[0])
                    COS_CD.append(given_pnr_df["COS_CD"].values[0])
                    OPER_OD_ORIG_CD.append(given_pnr_df["OPER_OD_ORIG_CD"].values[0])
                    OPER_OD_DEST_CD.append(given_pnr_df["OPER_OD_DEST_CD"].values[0])
                    DEP_KEY.append(given_pnr_df["DEP_KEY"].values[0])
                    DEP_DT.append(given_pnr_df["DEP_DT"].values[0])
                    ORIG_CD.append(given_pnr_df["ORIG_CD"].values[0])
                    DEST_CD.append(given_pnr_df["DEST_CD"].values[0])
                    FLT_NUM.append(given_pnr_df["FLT_NUM"].values[0])
                    DEP_DTML.append(given_pnr_df["DEP_DTML"].values[0])
                    ARR_DTML.append(given_pnr_df["ARR_DTML"].values[0])
                    DEP_DTMZ.append(given_pnr_df["DEP_DTMZ"].values[0])
                    ARR_DTMZ.append(given_pnr_df["ARR_DTMZ"].values[0])
                    PREV_OD_BROKEN_IND.append(given_pnr_df["OD_BROKEN_IND"].values[0])
                    PAX_CNT.append(given_pnr_df["PAX_CNT"].values[0])
                    CVM.append(given_pnr_df["CVM"].values[0])
                    PREV_CONN_TIME.append(given_pnr_df["CONN_TIME_MINS"].values[0])

                    # Now for the reaaccommodation
                    ALT_CABIN_CD.append("Either")
                    ALT_OPER_OD_ORIG_CD.append(leg_df["ORIG_CD"].values[0])
                    ALT_OPER_OD_DEST_CD.append(leg_df["DEST_CD"].values[0])
                    ALT_DEP_DT.append(leg_df["DEP_DT"].values[0])
                    ALT_ORIG_CD.append(leg_df["ORIG_CD"].values[0])
                    ALT_DEST_CD.append(leg_df["DEST_CD"].values[0])
                    ALT_FLT_NUM.append(leg_df["FLT_NUM"].values[0])
                    ALT_DEP_DTML.append(leg_df["DEP_DTML"].values[0])
                    ALT_ARR_DTML.append(leg_df["ARR_DTML"].values[0])
                    ALT_DEP_DTMZ.append(leg_df["DEP_DTMZ"].values[0])
                    ALT_ARR_DTMZ.append(leg_df["ARR_DTMZ"].values[0])
                
                
                # Now we change the connection time
                leg1, leg2 = reaccommodation[0], reaccommodation[1]
                leg1_df = available_flights_df[available_flights_df["DEP_KEY"] == leg1]
                leg2_df = available_flights_df[available_flights_df["DEP_KEY"] == leg1]
                
                ALT_CONN_TIME_MINS.append( None )

                first_time = pd.to_datetime(leg1_df["ARR_DTML"].values[0], errors="coerce", infer_datetime_format=True)
                second_time = pd.to_datetime(leg2_df["DEP_DTML"].values[0], errors="coerce", infer_datetime_format=True)
                delta_t = (first_time - second_time).total_seconds() / 60

                ALT_CONN_TIME_MINS.append(delta_t)

# Now we save the results into a dictionary

In [ ]:
Results_dict = {}

Results_dict["RECLOC"] = RECLOC
Results_dict["CABIN_CD"] = CABIN_CD
Results_dict["COS_CD"] = COS_CD
Results_dict["OPER_OD_ORIG_CD"] = OPER_OD_ORIG_CD
Results_dict["OPER_OD_DEST_CD"] = OPER_OD_DEST_CD
Results_dict["DEP_KEY"] = DEP_KEY
Results_dict["DEP_DT"] = DEP_DT
Results_dict["ORIG_CD"] = ORIG_CD
Results_dict["DEST_CD"] = DEST_CD
Results_dict["FLT_NUM"] = FLT_NUM
Results_dict["DEP_DTML"] = DEP_DTML
Results_dict["ARR_DTML"] = ARR_DTML
Results_dict["DEP_DTMZ"] = DEP_DTMZ
Results_dict["ARR_DTMZ"] = ARR_DTMZ
Results_dict["PREV_OD_BROKEN_IND"] = PREV_OD_BROKEN_IND
Results_dict["PAX_CNT"] = PAX_CNT
Results_dict["CVM"] = CVM
Results_dict["PREV_CONN_TIME"] = PREV_CONN_TIME

Results_dict["ALT_CABIN_CD"] = ALT_CABIN_CD
Results_dict["ALT_OPER_OD_ORIG_CD"] = ALT_OPER_OD_ORIG_CD
Results_dict["ALT_OPER_OD_DEST_CD"] = ALT_OPER_OD_DEST_CD
Results_dict["ALT_DEP_DT"] = ALT_DEP_DT
Results_dict["ALT_ORIG_CD"] = ALT_ORIG_CD
Results_dict["ALT_DEST_CD"] = ALT_DEST_CD
Results_dict["ALT_FLT_NUM"] = ALT_FLT_NUM
Results_dict["ALT_DEP_DTML"] = ALT_DEP_DTML
Results_dict["ALT_ARR_DTML"] = ALT_ARR_DTML
Results_dict["ALT_DEP_DTMZ"] = ALT_DEP_DTMZ
Results_dict["ALT_ARR_DTMZ"] = ALT_ARR_DTMZ
Results_dict["ALT_CONN_TIME_MINS"] = ALT_CONN_TIME_MINS

In [ ]:
Results_df = pd.DataFrame(Results_dict)
Results_df

# Save the results into a csv

In [ ]:
Results_df.to_csv("Results_DWAVE.csv", index = False)